# Práctica con alumnos · Pacientes y diabetes
**Laboratorio de Modelado** · Completá los `____` y los `TODO`. Ejecutá celda por celda y respondé las preguntas **Para discutir**.

Dataset: `datasets/pacientes_diabetes.csv` (1 200 pacientes, datos sintéticos). Un solo dataset, **dos preguntas**:
1. ¿Cuál será su nivel de **HbA1c** (hemoglobina glicosilada)? → regresión (`hba1c`).
2. ¿Tiene **diabetes**? → clasificación (`diabetes`, clase minoritaria ≈ 17 %).

⚠️ Acá la *accuracy* engaña: pensá qué error es más costoso.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from pathlib import Path
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import (mean_absolute_error, mean_squared_error, r2_score, confusion_matrix,
                             ConfusionMatrixDisplay, classification_report, roc_auc_score, RocCurveDisplay, log_loss)
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, StackingClassifier
from sklearn.neighbors import KNeighborsClassifier
sns.set_theme(style="whitegrid")
RANDOM_STATE = 42

## 1. Carga y exploración de datos
Primero miramos qué tenemos: filas, columnas, tipos, **valores faltantes** y cómo se distribuyen las variables.

In [ ]:
RUTA = Path("datasets/pacientes_diabetes.csv")
if not RUTA.exists():            # por si subís el CSV directo a Colab
    RUTA = Path("pacientes_diabetes.csv")
df = pd.read_csv(RUTA)
# TODO: mostrá las primeras filas, la info del DataFrame y los valores faltantes por columna
____

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 4))
sns.scatterplot(data=df, x="glucosa_ayunas", y="hba1c", alpha=.5, ax=ax[0]); ax[0].set_title("glucosa_ayunas vs hba1c")
sns.heatmap(df.corr(numeric_only=True), annot=True, fmt=".2f", cmap="RdBu_r", center=0, ax=ax[1]); ax[1].set_title("Correlaciones")
df["diabetes"].value_counts(normalize=True).plot.bar(ax=ax[2], color=["#2A9D8F", "#F2A541"]); ax[2].set_title("Proporción de clases: diabetes")
plt.tight_layout(); plt.show()

> **Para discutir:** ¿Qué variables parecen más relacionadas con `hba1c`? ¿Están balanceadas las clases de `diabetes`? ¿Qué consecuencia tiene eso para la *accuracy*?

## 2. Variables independientes (X), dependientes (y) y partición Train/Test
Tenemos **dos preguntas** sobre el mismo dataset:
- **Regresión** → ¿cuánto? → `hba1c` (continua, en % de hemoglobina glicosilada).
- **Clasificación** → ¿sí o no? → `diabetes` (binaria: paciente con diabetes).

Separamos un 20 % para el **test final** ("cofre bajo llave"). Con `stratify` conservamos la proporción de clases.

In [ ]:
X = df.drop(columns=____)            # ¿qué columnas NO son predictores?
y_reg, y_clf = df[____], df[____]
X_train, X_test, yr_train, yr_test, yc_train, yc_test = train_test_split(
    X, y_reg, y_clf, test_size=____, random_state=RANDOM_STATE, stratify=____)
print("Train:", X_train.shape, "| Test:", X_test.shape)
print("Proporción de clase positiva en train / test:", round(yc_train.mean(), 3), "/", round(yc_test.mean(), 3))

## 3. Regresión lineal: predecir un número
$\hat{y} = \beta_0 + \beta_1 x_1 + \dots + \beta_k x_k$

Usamos un **Pipeline** para encadenar *imputar → escalar → modelo*. Así el escalador y el imputador se ajustan **solo con train** (evita *data leakage*).

In [ ]:
reg = Pipeline([("imputer", SimpleImputer(strategy="median")),
                ("scaler", ____),
                ("modelo", ____)])
reg.____(X_train, yr_train)
pred = reg.____(X_test)
coefs = pd.Series(reg.named_steps["modelo"].coef_, index=X.columns).sort_values()
print("Intercepto (β0):", round(reg.named_steps["modelo"].intercept_, 2))
coefs.plot.barh(title="Coeficientes (variables estandarizadas)", color="#2A9D8F"); plt.show()

> **Para discutir:** ¿Qué signo tiene cada coeficiente y qué significa? ¿Cuál es la variable con mayor peso? ¿Por qué podemos compararlos entre sí?

In [ ]:
mae = ____(yr_test, pred)
mse = ____(yr_test, pred)
print(f"MAE={mae:.2f} | MSE={mse:.2f} | RMSE={____:.2f} | R²={____(yr_test, pred):.3f}")
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
lim = [yr_test.min(), yr_test.max()]
ax[0].scatter(yr_test, pred, alpha=.5); ax[0].plot(lim, lim, "r--"); ax[0].set(xlabel="Real", ylabel="Predicho", title="Real vs predicho")
ax[1].scatter(pred, ____, alpha=.5); ax[1].axhline(0, color="r", ls="--"); ax[1].set(xlabel="Predicho", ylabel="Residuo", title="Residuos")
plt.tight_layout(); plt.show()

## 4. ¿Cómo aprende? Descenso del gradiente "a mano"
Regresión simple con **una** variable estandarizada. Repetimos: predecir → medir el error (MSE) → mover $\beta_0,\beta_1$ en la dirección que baja la pérdida. `lr` es la **tasa de aprendizaje** (un hiperparámetro).

In [ ]:
d = df[["glucosa_ayunas", "hba1c"]].dropna()
x = ((d["glucosa_ayunas"] - d["glucosa_ayunas"].mean()) / d["glucosa_ayunas"].std()).values
t = d["hba1c"].values
b0, b1, lr, hist = 0.0, 0.0, 0.1, []
for epoch in range(60):
    err = (b0 + b1 * x) - t
    hist.append(np.mean(err ** 2))            # pérdida: MSE
    b0 -= lr * 2 * ____                       # TODO: gradiente respecto de β0  (pista: promedio del error)
    b1 -= lr * 2 * ____                       # TODO: gradiente respecto de β1  (pista: promedio de error * x)
chk = LinearRegression().fit(x.reshape(-1, 1), t)
print(f"Descenso del gradiente: β0={b0:.2f}  β1={b1:.2f}  | MSE final={hist[-1]:.1f}")
print(f"scikit-learn          : β0={chk.intercept_:.2f}  β1={chk.coef_[0]:.2f}")
plt.plot(hist); plt.xlabel("Iteración"); plt.ylabel("MSE"); plt.title("La pérdida baja en cada paso"); plt.show()

> **Probá:** cambiá `lr` a `0.001` y luego a `1.1`. ¿Qué pasa con la curva de pérdida? ¿Por qué?

## 5. Regresión logística: predecir una probabilidad
$p = \dfrac{1}{1+e^{-z}}$ con $z = \beta_0 + \beta_1 x_1 + \dots$ (función **sigmoide**). Pasamos de probabilidad a clase con un **umbral** (por defecto 0.5).

In [ ]:
clf_log = Pipeline([("imputer", SimpleImputer(strategy="median")),
                    ("scaler", StandardScaler()),
                    ("modelo", LogisticRegression(max_iter=1000))])
clf_log.fit(X_train, yc_train)
proba = clf_log.____(X_test)[:, 1]          # TODO: ¿qué método devuelve probabilidades?
print("Primeras 8 probabilidades:", proba[:8].round(2))
z = np.linspace(-8, 8, 200)
plt.plot(z, 1 / (1 + np.exp(-z))); plt.axhline(.5, color="gray", ls="--")
plt.title("Función sigmoide"); plt.xlabel("z = β0 + β1x1 + ..."); plt.ylabel("probabilidad"); plt.show()

In [ ]:
y_pred = (proba >= ____).astype(int)
ConfusionMatrixDisplay.from_predictions(yc_test, y_pred, cmap="Blues"); plt.title("Matriz de confusión (umbral 0.5)"); plt.show()
print(classification_report(____, ____, digits=3))
print("AUC:", round(roc_auc_score(____, ____), 3), "| Log Loss:", round(log_loss(yc_test, proba), 3))
print("Accuracy de un modelo 'tonto' que siempre dice 0:", round((yc_test == 0).mean(), 3))
RocCurveDisplay.from_predictions(yc_test, proba); plt.plot([0, 1], [0, 1], "k--"); plt.title("Curva ROC"); plt.show()

> **Para discutir:** Compará la *accuracy* con la del modelo "tonto". ¿Qué clase es la que más nos importa detectar? ¿Qué duele más: un falso positivo o un falso negativo?

### Efecto del umbral
Bajar el umbral atrapa más positivos (sube el **recall**) pero genera más falsas alarmas (baja la **precisión**).

In [ ]:
rows = []
for thr in [0.1, 0.2, 0.3, 0.5, 0.7]:
    yp = (proba >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(yc_test, yp).ravel()
    rows.append(dict(umbral=thr, TP=tp, FP=fp, FN=fn, TN=tn,
                     precision=____,            # TODO: TP / (TP + FP)
                     recall=____))              # TODO: TP / (TP + FN)
display(pd.DataFrame(rows).round(3))

## 6. Ensembles: Bagging, Boosting y Stacking
- **Bagging** (Random Forest): muchos árboles independientes → baja la *varianza*.
- **Boosting** (Gradient Boosting): árboles secuenciales que corrigen errores → baja el *sesgo*.
- **Stacking**: modelos de familias distintas + un meta-modelo que decide.

In [ ]:
imp = lambda: SimpleImputer(strategy="median")
modelos = {
    "LogReg": clf_log,
    "RandomForest (bagging)": Pipeline([("imp", imp()), ("m", RandomForestClassifier(n_estimators=____, random_state=RANDOM_STATE))]),
    "GradientBoosting (boosting)": Pipeline([("imp", imp()), ("m", ____(random_state=RANDOM_STATE))]),
    "Stacking": Pipeline([("imp", imp()), ("m", StackingClassifier(
        estimators=[("rf", RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE)),
                    ("knn", make_pipeline(StandardScaler(), KNeighborsClassifier())),
                    ("lr", make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)))],
        final_estimator=____))]),             # TODO: ¿qué meta-modelo usarías?
}
print("Modelos a comparar:", list(modelos))

## 7. Validación cruzada estratificada
Evaluamos cada modelo con **5 folds** sobre el set de *train* (el test sigue intacto). Miramos el **promedio** y la **desviación**: un buen modelo tiene promedio alto y desvío bajo.

In [ ]:
cv = StratifiedKFold(n_splits=____, shuffle=True, random_state=RANDOM_STATE)
res = {nombre: cross_val_score(m, X_train, yc_train, cv=cv, scoring="____") for nombre, m in modelos.items()}
# TODO: elegí el scoring ("f1", "recall", "roc_auc"...) pensando en qué error duele más
resumen = pd.DataFrame({k: {"Media": v.mean(), "Desvío": v.std()} for k, v in res.items()}).T.round(3)
display(resumen)
plt.figure(figsize=(9, 4)); sns.boxplot(data=pd.DataFrame(res), orient="h", palette="viridis")
plt.xlabel("Métrica por fold"); plt.show()

> **Para discutir:** ¿Hay un ganador claro? ¿Alguno es inestable (desvío alto)? ¿Vale la pena la complejidad del ensemble frente a la regresión logística?

## 8. Ajuste de hiperparámetros con GridSearchCV
Probamos **todas las combinaciones** de una grilla; cada una se evalúa con validación cruzada.

In [ ]:
pipe_rf = Pipeline([("imp", imp()), ("m", RandomForestClassifier(random_state=RANDOM_STATE))])
param_grid = {"m__n_estimators": [____, ____],        # TODO: completá la grilla (el prefijo m__ apunta al paso "m")
              "m__max_depth": [____, ____, None],
              "m__min_samples_leaf": [____, ____]}
gs = GridSearchCV(____, ____, cv=cv, scoring="____", n_jobs=-1)
gs.fit(X_train, yc_train)
print("Combinaciones:", len(gs.cv_results_["params"]), "x 5 folds =", len(gs.cv_results_["params"]) * 5, "entrenamientos")
print("best_params_:", gs.best_params_)
print("best_score_ :", round(gs.best_score_, 3))
display(pd.DataFrame(gs.cv_results_).sort_values("rank_test_score")[["params", "mean_test_score", "std_test_score"]].head(5).round(3))

## 9. Evaluación final: se abre el "cofre" **una sola vez**
Usamos `best_estimator_` (ya reentrenado con todo el train) y lo probamos contra el test que nunca vio.

In [ ]:
mejor = gs.____                                   # TODO: ¿qué atributo guarda el modelo ganador?
yp = mejor.predict(X_test); pp = mejor.predict_proba(X_test)[:, 1]
print(classification_report(yc_test, yp, digits=3))
print("AUC en test:", round(roc_auc_score(yc_test, pp), 3))
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ConfusionMatrixDisplay.from_predictions(yc_test, yp, cmap="Blues", ax=ax[0]); ax[0].set_title("Matriz de confusión (test)")
pd.Series(mejor.named_steps["m"].feature_importances_, index=X.columns).sort_values().plot.barh(ax=ax[1], color="#2A9D8F")
ax[1].set_title("Importancia de variables (Random Forest)"); plt.tight_layout(); plt.show()